# Enterprise Tools Mcp

## Setup
Use the separate engineering environment in README.md. Read the backend primer first. These assignments use local services and your own files; no model/API calls are made by default. All records and keys are invented teaching examples. Keep the whole track folder. Edit the numbered TODOs in `submission.py`, then run `importlib.reload(impl)` before rerunning tests. Instructor implementations are not included in the student handout.

Use a two-pass workflow: attempt each small TODO using the contract, then consult the staged hints in HINTS.md if needed. Record how much help you used in your learning log. Keep your solution in your own Git branch or copy. Never replace it with the instructor implementation before attempting the exercise.

In [1]:
from pathlib import Path
import sys, importlib, json
HERE = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "engineeringlab").is_dir() or (p / "labs/engineering/engineeringlab").is_dir())
TRACK = HERE if (HERE / "engineeringlab").is_dir() else HERE / "labs/engineering"
if str(TRACK) not in sys.path: sys.path.insert(0, str(TRACK))
RUN_EXERCISES = False
MODULE = "submission"
impl = importlib.import_module(MODULE)
from engineeringlab import checks
print("Implement submission.py, reload it, then enable exercise checks. Default execution is not a pass.")

Implement submission.py, reload it, then enable exercise checks. Default execution is not a pass.


## Goal
Implement three small boundaries: authorization, atomic ticket creation with replay protection, and MCP tools forwarding to an HTTP service. Prerequisites: Labs 5, 8, 15–16; SQL transactions and HTTP basics from the primer. Plan two study sessions (about 3–4 hours total, an unpiloted estimate).

The system has three identities: the API caller, the tenant selected by its key, and a separate reviewer. A model can propose an operation; only the reviewer endpoint can create its approval receipt. The receipt is bound to tenant, operation ID, resource, and summary. Reusing the ID with changed content is a conflict. The local keys demonstrate identity separation; they are public and do not make this a production authentication system.

The action ledger uses SQLite to make this exercise portable. Your RAG corpus remains in pgvector from Labs 12–16; the capstone combines the services.

## Steps
### 1. Trace the provided transport before editing it
Read `engineeringlab/support.py`. Draw the path from X-API-Key to tenant, allowed resources, approval lookup, authorization, and transaction. Find where a caller-supplied tenant or approval field is rejected by the request schema. Explain why the MCP server must never receive the reviewer key.

A transaction groups database changes. The idempotency key identifies one intended business operation. Here the composite key is `(tenant, operation_id)` so another tenant's operation cannot be treated as a replay. The stored payload lets the service distinguish a genuine retry from a changed request.

### 2. TODO 18.1 — authorize
Edit `authorize`: accept only an actual boolean True and a resource in the trusted resource set. Reject strings such as "yes" and integer 1. The approved value comes from a server-side receipt lookup, not from model arguments.

In [2]:
if RUN_EXERCISES:
    importlib.reload(impl)
    checks.authorization(impl)
    print("Authorization contract passed")

### 3. TODO 18.2 — commit or replay one ticket
Implement the transaction in `commit_ticket`. Start with BEGIN IMMEDIATE, select by both identity columns, and compare the canonical payload. On an identical retry return the old ID. On changed content raise Conflict. For a new operation insert one row and commit. Roll back on every exception. Use explicit INSERT column names so an additive migration can remain compatible.

Return `{ticket_id: str, replayed: bool}`. Add your own concurrent duplicate-request test. A generated UUID alone is not replay protection. A lost HTTP response also does not imply that the database write failed.

In [3]:
if RUN_EXERCISES:
    importlib.reload(impl)
    checks.transactions(impl)
    print("Replay, conflict, tenant separation, and reopened-connection checks passed")

### 4. TODO 18.3 — expose an actual MCP server
Implement `register_tools` using the installed MCP SDK's `server.tool()` decorator. Expose create_ticket(operation_id, resource, summary) and ticket_status(operation_id). Use only the supplied HTTP client; it holds the caller key and timeout. Forward errors with raise_for_status. No tool accepts a tenant, API key, or approval flag.

The next check launches a real Uvicorn service and an MCP subprocess over stdio, creates approval through a separate reviewer request, and invokes your tool through an MCP client. It also checks that an unapproved call creates no ticket. The subprocess stdout is the protocol channel; write diagnostics to stderr. This local stdio exercise does not implement remote MCP OAuth.

In [4]:
if RUN_EXERCISES:
    importlib.reload(impl)
    result = await checks.mcp_roundtrip(MODULE)
    print(result)

## Checks and submission
Run `python -m pytest tests -q -k "authorization or transactions or mcp"` from this track after completing TODOs. Add a changed-summary-after-approval case, two simultaneous retries, a wrong-resource request, and a missing API key. For response-loss practice, discard a successful response deliberately, query by operation ID, then retry and verify the same ticket ID. State that deliberate response discard simulates this failure; do not claim a physical network outage.

Submit code, your identity/transaction diagram, tests, and one trace showing why an action was allowed or denied. Rubric: authorization 20, atomic commit 35, MCP integration 25, added tests and explanation 20. Default notebook execution is setup only; all checks must be enabled for assessment.

## Next Steps
Lab 19 packages this service and evolves its schema. Source: [official MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk), [SQLite transactions](https://www.sqlite.org/lang_transaction.html), [FastAPI dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/). Interfaces checked 2026-10-08 against the recorded package versions.